In [1]:
"""
MDB Financing — A Hands-On Math Primer
======================================
A small, dependency-free toolkit illustrating the financial mathematics
embedded in Multilateral Development Bank (MDB) instruments, plus a simple
IMF rate-of-charge calculator for contrast.

Pure standard library — just run:  python3 mdb_finance_primer.py

Concepts covered
----------------
1. Funding-to-lending spread (the MDB intermediation engine)
2. Grant element  (concessionality of a loan)  -- DAC-style + bullet loan
3. Bond pricing and yield-to-maturity (how MDBs raise money)
4. Blended-finance leverage / mobilization ratio
5. IMF rate of charge (SDR rate + margin + surcharges) -- the contrast

NOTE ON CONVENTIONS: real DAC grant-element calculations use semiannual
payments and differentiated discount rates by income group. Here the
discount rate and payment frequency are parameters so you can match
whatever convention you need.
"""

from __future__ import annotations


# ---------------------------------------------------------------------------
# 1. The intermediation engine: funding cost vs lending rate
# ---------------------------------------------------------------------------
def funding_spread(bond_yield: float, lending_rate: float) -> float:
    """Gross margin the MDB earns: it borrows at `bond_yield`, lends higher.

    All rates are annual decimals (0.04 == 4%). Returns the spread in bps.
    """
    return (lending_rate - bond_yield) * 10_000  # basis points


# ---------------------------------------------------------------------------
# 2. Grant element: how much of a "loan" is really a gift
# ---------------------------------------------------------------------------
def grant_element_amortizing(
    face: float,
    rate: float,
    grace_years: float,
    maturity_years: float,
    discount: float,
    freq: int = 2,
) -> float:
    """Grant element (%) for a loan with a grace period and equal principal
    repayments after grace. Interest is paid (not capitalized) during grace.

    face          : principal / face value
    rate          : annual interest rate on the loan (decimal)
    grace_years   : years of grace (principal deferred, interest still paid)
    maturity_years: total tenor in years
    discount      : reference discount rate (decimal) e.g. 0.05 or 0.10
    freq          : payments per year (DAC convention = 2)

    GE > 0 when rate < discount. Higher grace/maturity -> higher GE.
    """
    n_total = round(maturity_years * freq)
    n_grace = round(grace_years * freq)
    n_repay = n_total - n_grace
    per_rate = rate / freq
    per_disc = discount / freq
    principal_pmt = face / n_repay

    balance = face
    pv_debt_service = 0.0
    for t in range(1, n_total + 1):
        interest = balance * per_rate
        principal = principal_pmt if t > n_grace else 0.0
        debt_service = interest + principal
        pv_debt_service += debt_service / (1 + per_disc) ** t
        balance -= principal

    return (1 - pv_debt_service / face) * 100


def grant_element_bullet(
    face: float, rate: float, maturity_years: int, discount: float
) -> float:
    """Grant element (%) for a simple bullet loan: interest annually, full
    principal repaid at maturity. Matches the hand-worked chat example.
    """
    annuity_factor = (1 - (1 + discount) ** -maturity_years) / discount
    pv_interest = face * rate * annuity_factor
    pv_principal = face / (1 + discount) ** maturity_years
    pv_debt_service = pv_interest + pv_principal
    return (1 - pv_debt_service / face) * 100


# ---------------------------------------------------------------------------
# 3. Bond pricing & yield-to-maturity (the MDB funding side)
# ---------------------------------------------------------------------------
def bond_price(
    face: float, coupon_rate: float, years: int, ytm: float, freq: int = 2
) -> float:
    """Clean price of a coupon bond = PV of coupons + PV of face."""
    n = years * freq
    coupon = face * coupon_rate / freq
    per_ytm = ytm / freq
    pv_coupons = sum(coupon / (1 + per_ytm) ** t for t in range(1, n + 1))
    pv_face = face / (1 + per_ytm) ** n
    return pv_coupons + pv_face


def bond_ytm(
    price: float, face: float, coupon_rate: float, years: int, freq: int = 2
) -> float:
    """Solve for yield-to-maturity by bisection. Returns annual decimal."""
    lo, hi = 1e-6, 1.0  # 0% to 100%
    for _ in range(200):
        mid = (lo + hi) / 2
        if bond_price(face, coupon_rate, years, mid, freq) > price:
            lo = mid  # price too high -> need higher yield
        else:
            hi = mid
    return (lo + hi) / 2


# ---------------------------------------------------------------------------
# 4. Blended-finance leverage / mobilization ratio
# ---------------------------------------------------------------------------
def mobilization_ratio(public_input: float, total_capital: float) -> float:
    """How many dollars of total capital each public dollar mobilizes.
    private_mobilized = total_capital - public_input
    ratio = private_mobilized / public_input
    """
    private_mobilized = total_capital - public_input
    return private_mobilized / public_input


# ---------------------------------------------------------------------------
# 5. IMF rate of charge (the contrast: monetary, not development, finance)
# ---------------------------------------------------------------------------
def imf_rate_of_charge(
    sdr_rate: float,
    margin: float = 0.01,
    level_surcharge: float = 0.0,
    time_surcharge: float = 0.0,
) -> float:
    """Illustrative IMF basic rate of charge.

    The IMF prices off the SDR interest rate plus a margin (recently ~100 bps),
    with additional surcharges for large (level) or prolonged (time) borrowing.
    Unlike an MDB it is NOT funding this from bond issuance — it draws on the
    pooled quota resources of its members.
    """
    return sdr_rate + margin + level_surcharge + time_surcharge


# ---------------------------------------------------------------------------
# Worked examples
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    line = "-" * 68

    print(line)
    print("1. FUNDING-TO-LENDING SPREAD (the intermediation engine)")
    print(line)
    by, lr = 0.035, 0.052  # MDB issues bonds at 3.5%, lends at 5.2%
    print(f"  Bond yield (cost of funds): {by:.2%}")
    print(f"  Lending rate to borrower:   {lr:.2%}")
    print(f"  Gross spread:               {funding_spread(by, lr):.0f} bps")

    print()
    print(line)
    print("2. GRANT ELEMENT (concessionality)")
    print(line)
    # Bullet loan matching the hand-worked chat example
    ge_bullet = grant_element_bullet(100, 0.02, 10, 0.06)
    print("  Bullet loan: $100M, 2% rate, 10y, repaid at maturity, disc 6%")
    print(f"    Grant element: {ge_bullet:.1f}%   (~29 cents/$ is a gift)")
    print()
    # IDA-style highly concessional loan
    ge_ida = grant_element_amortizing(
        face=100, rate=0.01, grace_years=10, maturity_years=40,
        discount=0.05, freq=2,
    )
    print("  IDA-style: $100M, 1% rate, 10y grace, 40y maturity, disc 5%")
    print(f"    Grant element: {ge_ida:.1f}%")
    print()
    # Near-market loan: little to no concessionality
    ge_market = grant_element_amortizing(
        face=100, rate=0.05, grace_years=0, maturity_years=15,
        discount=0.05, freq=2,
    )
    print("  Near-market: $100M, 5% rate, no grace, 15y, disc 5%")
    print(f"    Grant element: {ge_market:.1f}%   (~0 -> not concessional)")

    print()
    print(line)
    print("3. BOND PRICING & YIELD (how the MDB raises money)")
    print(line)
    price = bond_price(face=1000, coupon_rate=0.03, years=10, ytm=0.035)
    print(f"  Price of a 10y, 3% coupon, $1000 bond at 3.5% YTM: ${price:.2f}")
    print("  (Trades below par because coupon < market yield.)")
    implied = bond_ytm(price=950, face=1000, coupon_rate=0.03, years=10)
    print(f"  If that bond instead trades at $950, implied YTM: {implied:.3%}")

    print()
    print(line)
    print("4. BLENDED-FINANCE LEVERAGE")
    print(line)
    ratio = mobilization_ratio(public_input=20, total_capital=100)
    print("  $20M concessional input inside a $100M project")
    print(f"    Mobilization ratio: {ratio:.1f}x private capital per public $")

    print()
    print(line)
    print("5. IMF RATE OF CHARGE (contrast: monetary, quota-funded)")
    print(line)
    basic = imf_rate_of_charge(sdr_rate=0.03, margin=0.01)
    surcharged = imf_rate_of_charge(
        sdr_rate=0.03, margin=0.01, level_surcharge=0.02, time_surcharge=0.01
    )
    print(f"  SDR rate 3% + 100 bps margin:        {basic:.2%}  (basic)")
    print(f"  + level + time surcharges:           {surcharged:.2%}  (large/long use)")
    print()
    print("  Note: the IMF funds this from POOLED MEMBER QUOTAS + SDRs,")
    print("  not from issuing bonds the way an MDB does. Different machine.")
    print(line)

--------------------------------------------------------------------
1. FUNDING-TO-LENDING SPREAD (the intermediation engine)
--------------------------------------------------------------------
  Bond yield (cost of funds): 3.50%
  Lending rate to borrower:   5.20%
  Gross spread:               170 bps

--------------------------------------------------------------------
2. GRANT ELEMENT (concessionality)
--------------------------------------------------------------------
  Bullet loan: $100M, 2% rate, 10y, repaid at maturity, disc 6%
    Grant element: 29.4%   (~29 cents/$ is a gift)

  IDA-style: $100M, 1% rate, 10y grace, 40y maturity, disc 5%
    Grant element: 54.8%

  Near-market: $100M, 5% rate, no grace, 15y, disc 5%
    Grant element: -0.0%   (~0 -> not concessional)

--------------------------------------------------------------------
3. BOND PRICING & YIELD (how the MDB raises money)
--------------------------------------------------------------------
  Price of a 10y, 3% 

In [5]:
def npv_flat(cf, n, i):
    return sum(cf / (1 + i) ** n for n in range(1, n + 1))

In [12]:
interest = npv_flat(2, 10, 0.06)
print(f"The interest to be paid is ${interest:.2f}M.")

The interest to be paid is $14.72M.


In [13]:
loan_amt = 100
i = 0.06
n = 10
principal = loan_amt / (1 + i) ** n
print(f"The present value of the loan amount is ${principal:.2f}M.")

The present value of the loan amount is $55.84M.


In [15]:
debt_service = interest + principal
print(f"The total debt service is ${debt_service:.2f}M.")

The total debt service is $70.56M.


In [18]:
grant_element = (1 - debt_service / loan_amt) * 100
print(f"The grant element of this loan is {grant_element:.2f}%.")

The grant element of this loan is 29.44%.
